## Train / Validation / Test Split

In [1]:
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("../data/metadata.csv")

train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)

print(len(train_df))
print(len(val_df))
print(len(test_df))

11661
2499
2499


## Kiểm tra Class Balance Sau Split

In [2]:
print(train_df["label"].value_counts())

print(val_df["label"].value_counts())

print(test_df["label"].value_counts())

label
OtherRicePests     1902
Blast              1583
Healthy            1576
BrownSpot          1534
Planthopper        1303
BacterialBlight    1140
DeadHeart           942
StemBorer           631
LeafDamage          612
SheathBlight        438
Name: count, dtype: int64
label
OtherRicePests     407
Blast              339
Healthy            338
BrownSpot          329
Planthopper        279
BacterialBlight    245
DeadHeart          202
StemBorer          136
LeafDamage         131
SheathBlight        93
Name: count, dtype: int64
label
OtherRicePests     408
Blast              339
Healthy            337
BrownSpot          329
Planthopper        280
BacterialBlight    244
DeadHeart          202
StemBorer          135
LeafDamage         131
SheathBlight        94
Name: count, dtype: int64


## Tạo Dataset Folder

In [3]:
import os
import shutil

base_output = r"D:\KÌ 7\RiceDataset_Split"

for split_name, split_df in [
    ("train", train_df),
    ("val", val_df),
    ("test", test_df)
]:

    for _, row in split_df.iterrows():

        src = row["image_path"]

        label = row["label"]

        dst_folder = os.path.join(
            base_output,
            split_name,
            label
        )

        os.makedirs(dst_folder, exist_ok=True)

        shutil.copy2(
            src,
            os.path.join(
                dst_folder,
                os.path.basename(src)
            )
        )

print("Done")

Done


## Data Augmentation

### Train Transform


In [4]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224,224)),

    transforms.RandomHorizontalFlip(),

    transforms.RandomRotation(20),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

### Validation Transform

In [5]:
val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

### Test Transform

In [6]:
test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225]
    )
])

## Tính Class Weight

In [7]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(df["label"])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=df["label"]
)

print(weights)

[1.02265193 0.73679788 0.75999088 1.23766716 0.74007108 1.90606407
 0.61313949 0.89468314 2.66544    1.84689579]


## Tạo DataLoader

In [8]:
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

train_dataset = ImageFolder(
    r"D:\KÌ 7\RiceDataset_Split\train",
    transform=train_transform
)

val_dataset = ImageFolder(
    r"D:\KÌ 7\RiceDataset_Split\val",
    transform=val_transform
)

test_dataset = ImageFolder(
    r"D:\KÌ 7\RiceDataset_Split\test",
    transform=test_transform
)

In [9]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [10]:
import numpy as np

np.save(
    "../data/class_weights.npy",
    weights
)